# <p style="background-color:#EFE4B0;font-family:Georgia;color:#990F02;font-size:150%;font-weight:bold;text-align:center;border-radius:10px 10px;">Прогноз кассовых сборов по данным из TMDB</p>
<center>
  <div style="display: flex; justify-content: center; align-items: center; gap: 20px;">
    <img src="https://pikuco.ru/upload/test_stable/576/5767eef7c34da2031fcfcb102176088f.webp"
         width="1200" 
         height="400">
    <img src="https://kodi.tv/images/addons/omega/context.embuary.info/resources/icon.png"
         width="200" 
         height="400">
  </div>
</center>

### <p style="margin-top:20px; background-color:#EFE4B0; background-size:50%; font-weight:bold; font-family:Georgia;color:#990F02;font-size:120%;border-radius:5px 5px;display:inline-block">Описание набора данных</p>

<p style="font-family:Georgia;color:#990F02;font-size:120%; margin-top: 10px"> Данные были получены с платформы <a href="https://www.kaggle.com/competitions/tmdb-box-office-prediction/data" style="color:#228B22;">Kaggle</a>.
    <br>В этом наборе данных представлено 7398 фильмов и разнообразные метаданные, полученные из <a href="https://www.themoviedb.org/documentation/api" style="color:#228B22;">The Movie Database (TMDB)</a>. Фильмы имеют идентификаторы (id). Включенные данные содержат информацию о составе актеров, съемочной группе, ключевых словах сюжета, бюджете, постерах, датах выхода, языках, производственных компаниях и странах.Нам предстоит предсказать мировую кассу для 4398 фильмов из тестового файла.
<br><br><span style="font-weight:bold">Примечание</span> - многие фильмы переснимаются с течением времени, поэтому может показаться, что несколько экземпляров одного и того же фильма присутствуют в данных, однако это разные фильмы, и их следует рассматривать как отдельные. Кроме того, некоторые фильмы могут иметь одинаковое название, но быть совершенно не связанными.Например, «Каратэ-пацан» (id: 5266) был выпущен в 1986 году, в то время как явно (или, возможно, субъективно) менее удачный ремейк (id: 1987) вышел в 2010 году. Также, хотя «Холодное сердце» (id: 5295), выпущенное Disney в 2013 году, может быть широко известным названием, не забывайте о менее популярном фильме «Холодное сердце» (id: 139), выпущенном на три года раньше, рассказывающем о лыжниках, застрявших на подъемнике…</p>

## Loading libraries

In [1]:
import pandas as pd
import numpy as np
import ast
from typing import *
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import plotly.express as px
from PIL import Image
from urllib.request import urlopen
from matplotlib.colors import ListedColormap
import seaborn as sns
from pprint import pprint
from collections import Counter 
from wordcloud import WordCloud
from time import sleep

import warnings
warnings.filterwarnings("ignore")

In [58]:
raw_data = pd.read_csv("data/TMDb_data_from_1990-2025.zip", index_col=0)

In [59]:
# Скопируем данныечтобы под рукой всегда были сырые данные
data = raw_data.copy()
data.head()

,id,belongs_to_collection,budget,genres,homepage,imdb_id,original_language,original_title,overview,popularity,...,release_date,runtime,spoken_languages,status,tagline,title,keywords,cast,crew,revenue
0,1310860,NaN,1200000.000000,[],NaN,tt33888380,ar,عودة الهارب,NaN,2.730100,...,1990-01-01 00:00:00,100.000000,"[{'english_name': 'Arabic', 'iso_639_1': 'ar',...",Released,NaN,عودة الهارب,[],"[{'adult': False, 'gender': 2, 'id': 129473, '...","[{'adult': False, 'gender': 2, 'id': 4802954, ...",3400000.000000
1,60778,NaN,0.000000,"[{'id': 28, 'name': 'боевик'}, {'id': 35, 'nam...",NaN,tt0099460,en,Downtown,Молодого неопытного полицейского переводят из ...,1.869400,...,1990-01-12 00:00:00,96.000000,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,NaN,Крутой район,[],"[{'adult': False, 'gender': 2, 'id': 11085, 'k...","[{'adult': False, 'gender': 0, 'id': 31025, 'k...",2346150.000000
2,25018,"{'id': 111751, 'name': 'Техасская резня бензоп...",2000000.000000,"[{'id': 27, 'name': 'ужасы'}]",NaN,tt0099994,en,Leatherface: The Texas Chainsaw Massacre III,"Два студента, едущие на машине от одного побер...",8.221500,...,1990-01-12 00:00:00,81.000000,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,NaN,Техасская резня бензопилой 3: Кожаное лицо,"[{'id': 9663, 'name': 'sequel'}, {'id': 11545,...","[{'adult': False, 'gender': 1, 'id': 151363, '...","[{'adult': False, 'gender': 2, 'id': 10051, 'k...",5765562.000000
3,22585,NaN,0.000000,"[{'id': 35, 'name': 'комедия'}, {'id': 28, 'na...",NaN,tt0100631,en,Ski Patrol,"Главные герои - кучка недотеп и болванов, вызы...",1.434600,...,1990-01-12 00:00:00,91.000000,"[{'english_name': 'French', 'iso_639_1': 'fr',...",Released,NaN,Лыжный патруль,"[{'id': 6075, 'name': 'sports'}, {'id': 159558...","[{'adult': False, 'gender': 2, 'id': 8049, 'kn...","[{'adult': False, 'gender': 2, 'id': 152315, '...",8533973.000000
4,11060,NaN,15000000.000000,"[{'id': 80, 'name': 'криминал'}, {'id': 18, 'n...",NaN,tt0099850,en,Internal Affairs,"Когда любой из нас попадает в беду, страж прав...",6.019100,...,1990-01-12 00:00:00,114.000000,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,NaN,Внутреннее расследование,"[{'id': 4654, 'name': 'undercover agent'}, {'i...","[{'adult': False, 'gender': 2, 'id': 1205, 'kn...","[{'adult': False, 'gender': 2, 'id': 6111, 'kn...",27734391.000000


## Overview

In [23]:
fontdict={"fontfamily": "arial","color": "#682F2F"}
cmap = ListedColormap(["#682F2F", "#9E726F", "#D6B2B1", "#B9C0C9", "#9F8A78", "#F3AB60", "#EFE4B0"])

In [60]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 19364 entries, 0 to 19363
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id                     19364 non-null  int64  
 1   belongs_to_collection  3307 non-null   object 
 2   budget                 19364 non-null  float64
 3   genres                 19364 non-null  object 
 4   homepage               271 non-null    object 
 5   imdb_id                17325 non-null  object 
 6   original_language      19364 non-null  object 
 7   original_title         19363 non-null  object 
 8   overview               12140 non-null  object 
 9   popularity             19364 non-null  float64
 10  poster_path            18953 non-null  object 
 11  production_companies   19364 non-null  object 
 12  production_countries   19364 non-null  object 
 13  release_date           19364 non-null  object 
 14  runtime                19364 non-null  float64
 15  spoken_

In [25]:
pd.set_option("display.float_format", "{:.6f}".format)
data.describe(include=['int64', 'float64'], )

,id,budget,popularity,runtime,revenue
count,19364.000000,19364.000000,19364.000000,19364.000000,19364.000000
mean,422350.967724,12930005.504131,3.115028,93.957757,36904948.941799
std,453759.006847,31157037.373370,8.598468,40.032565,120811414.066394
min,5.000000,0.000000,0.000000,0.000000,-12.000000
25%,30862.500000,0.000000,0.963975,87.000000,67112.500000
50%,264650.000000,10000.000000,2.398000,99.000000,1874743.500000
75%,688029.250000,11000000.000000,3.681725,113.000000,18134536.500000
max,1559917.000000,583900000.000000,590.559100,800.000000,2923706026.000000


In [54]:
# data['release_date'] = pd.to_datetime(data['release_date'], format="mixed")
data.iloc[-100:-50, :]['release_date']

19264    2025-09-04
19265    2025-09-19
19266    2025-09-08
19267    2025-09-26
19268    2025-10-08
19269    2025-12-10
19270    2025-12-24
19271    2025-11-29
19272    2025-11-30
19273    2006-02-07
19274    1999-02-17
19275    1993-09-10
19276    1991-11-10
19277    1991-02-07
19278    2008-01-01
19279    1999-01-03
19280    2007-02-12
19281    2007-08-03
19282    2005-01-01
19283    2008-12-04
19284    1968-04-03
19285    2008-11-07
19286    2007-06-20
19287    2007-04-08
19288    2009-03-13
19289    1982-12-17
19290    2000-11-09
19291    1995-05-19
19292    2004-11-24
19293    1996-01-26
19294    2003-03-22
19295    2006-03-01
19296    1973-10-25
19297    2006-09-09
19298    2004-08-10
19299    1969-04-24
19300    2007-11-08
19301    1999-03-26
19302    2000-01-01
19303    1981-07-24
19304    2008-01-01
19305    1951-09-14
19306    2009-06-03
19307    1985-06-01
19308    2007-01-01
19309    2009-05-27
19310    2009-05-20
19311    2007-05-18
19312    1981-08-13
19313    2008-09-12
